In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cnc_clean.csv")
print(df.shape)
df.head()

(2000, 9)


,part_id,material_type,spindle_speed_rpm,feed_rate_mm_min,cutting_temp_c,vibration_mm_s,tool_wear_min,coolant_flow_lpm,defect
0,1,Steel,1237.0,213.9,203.4,3.90,223.8,7.28,1
1,2,Titanium,1252.0,300.5,204.8,2.93,91.9,7.93,0
2,3,Aluminium,1432.0,126.4,166.7,3.62,175.1,10.01,0
3,4,Aluminium,1610.0,198.7,169.0,2.49,4.9,5.10,0
4,5,Steel,1774.0,225.6,190.6,2.46,49.2,8.01,0


In [3]:
fe = df.copy()

fe["cutting_intensity"] = fe["spindle_speed_rpm"] * fe["feed_rate_mm_min"] / 1000
fe["wear_x_vibration"] = fe["tool_wear_min"] * fe["vibration_mm_s"]
fe["heat_per_coolant"] = fe["cutting_temp_c"] / fe["coolant_flow_lpm"]
fe["vibration_per_speed"] = fe["vibration_mm_s"] / fe["spindle_speed_rpm"] * 1000
fe[["cutting_intensity", "wear_x_vibration","heat_per_coolant", "vibration_per_speed"]].describe().round(2)

,cutting_intensity,wear_x_vibration,heat_per_coolant,vibration_per_speed
count,2000.00,2000.00,2000.00,2000.00
mean,298.13,389.23,24.10,2.08
std,84.93,277.78,7.06,0.53
min,86.89,0.18,11.16,0.92
25%,236.04,147.62,19.71,1.69
50%,291.85,343.17,22.85,2.03
75%,351.23,598.35,27.14,2.38
max,657.64,1181.37,94.17,4.86


In [8]:
cols = ["tool_wear_min", "vibration_mm_s", "cutting_temp_c",
        "cutting_intensity", "wear_x_vibration",
        "heat_per_coolant", "vibration_per_speed"]

fe[cols + ["defect"]].corr()["defect"].drop("defect").sort_values(ascending=False).round(3)

wear_x_vibration       0.590
vibration_mm_s         0.556
cutting_temp_c         0.555
tool_wear_min          0.538
heat_per_coolant       0.361
vibration_per_speed    0.326
cutting_intensity      0.169
Name: defect, dtype: float64

In [9]:
fe = pd.get_dummies(fe, columns=["material_type"], drop_first=True, dtype=int)
fe.head()

,part_id,spindle_speed_rpm,feed_rate_mm_min,cutting_temp_c,vibration_mm_s,tool_wear_min,coolant_flow_lpm,defect,cutting_intensity,wear_x_vibration,heat_per_coolant,vibration_per_speed,material_type_Steel,material_type_Titanium
0,1,1237.0,213.9,203.4,3.90,223.8,7.28,1,264.5943,872.820,27.939560,3.152789,1,0
1,2,1252.0,300.5,204.8,2.93,91.9,7.93,0,376.2260,269.267,25.825977,2.340256,0,1
2,3,1432.0,126.4,166.7,3.62,175.1,10.01,0,181.0048,633.862,16.653347,2.527933,0,0
3,4,1610.0,198.7,169.0,2.49,4.9,5.10,0,319.9070,12.201,33.137255,1.546584,0,0
4,5,1774.0,225.6,190.6,2.46,49.2,8.01,0,400.2144,121.032,23.795256,1.386697,1,0


In [ ]:
fe = fe.drop(columns=["part_id"])    

print("Final shape:", fe.shape)
print(fe.columns.tolist())

fe.to_csv("../data/processed/cnc_features.csv", index=False)
print("Saved ✅")

Final shape: (2000, 13)
['spindle_speed_rpm', 'feed_rate_mm_min', 'cutting_temp_c', 'vibration_mm_s', 'tool_wear_min', 'coolant_flow_lpm', 'defect', 'cutting_intensity', 'wear_x_vibration', 'heat_per_coolant', 'vibration_per_speed', 'material_type_Steel', 'material_type_Titanium']
Saved ✅
